# Dates: tool calling and LLM reasoning

Two dates are extracted, normalised to ISO format, and classified against a
fixed reference date of 2024-01-01.

## Results

| Date | Page | As written | Normalised | Status |
|---|---|---|---|---|
| Distribution | 1 | 16 February 2024 | 2024-02-16 | Upcoming |
| Estate Duty | 36 | 15 February 2008 | 2008-02-15 | Expired |

## Who does what

| Job | Done by | Why |
|---|---|---|
| Select pages | Code | `date_pages` binds each date to a page - 1 and 36 - before extraction starts |
| Find a date in prose | LLM | Fuzzy: the date sits in a sentence, phrased variably |
| Parse it to ISO | Tool, via MCP | Deterministic: one right answer |
| Classify against a reference | LLM | The design calls for reasoning |
| Check that classification | Tool | Catches a confidently wrong answer |

The model handles what needs judgement; the tool handles what has a single
correct answer. Step 2 inverts this deliberately - the LLM classifies because
that is the intent, so the tool becomes a checker rather than the
answer.

Assumptions are at the end.

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.config import load_config
from src.evaluation import score_dates
from src.extraction.date_reasoning import check, classify, to_output
from src.extraction.dates import (
    find_dates,
    normalize_dates,
    normalize_dates_mcp,
    normalized_with_context,
)
from src.ingestion.download import ensure_pdf
from src.ingestion.parser import extract_pages
from src.tools.mcp_client import list_tools

config = load_config(Path.cwd().parent / "config.yml")
pdf_path = ensure_pdf(config.pdf_url)

print(f"provider={config.provider}  model={config.model}")
print(f"date pages={config.date_pages}")

provider=groq  model=llama-3.1-8b-instant
date pages=[1, 36]


## The local MCP server

A local MCP server is the primary path, with a tool decorator as fallback.
Both exist here and share one implementation: `mcp_server.py` imports from
`date_tool.py` rather than reimplementing it.

"Local" means stdio - the client launches the server as a subprocess and they
exchange JSON-RPC on stdin/stdout. Nothing is networked or deployed, and the
server exists only for the lifetime of the client that started it.

In [2]:
# Starts the server as a subprocess, lists its tools, shuts it down.
for tool in await list_tools():
    print(f"{tool['name']}")
    print(f"    {tool['description'].splitlines()[0]}")

normalize_date
    Convert a date written in prose into ISO format (YYYY-MM-DD).


## Step 1: find and normalise

### The input

Only pages 1 and 36 are read. Other dates appear elsewhere in the document, so
narrowing the input removes the ambiguity before the model sees it - the same
page-binding used in Part 1.

In [3]:
page_text = extract_pages(pdf_path, config.date_page_numbers)

for line in page_text.splitlines():
    if line.startswith("--- page") or "February" in line:
        print(line.strip()[:120])

--- page 1 ---
Distributed on Budget Day: 16 February 2024
--- page 36 ---
person who dies after 15 February 2008.


### The model finds the dates

It returns each date **as written**, with the sentence it came from. It does not
convert anything - that is the tool's job.

This is where an LLM earns its place over a regex. A pattern would match these
two dates today, but every new phrasing or format needs another pattern, and
the set grows with each document.

In [ ]:
found = find_dates(pdf_path, config=config)

for finding in (found.distribution, found.estate_duty):
    print(f"page {finding.page}: {finding.date_as_written!r}")
    print(f"    from: {finding.original_text[:90]}")

### The tool normalises them

Over MCP: the client starts the server, calls `normalize_date` for each date,
and shuts it down. The in-process `@tool` path is shown alongside to confirm
both routes agree.

In [5]:
via_mcp = await normalize_dates_mcp(found)
via_tool = normalize_dates(found)

print(f"via MCP server:  {via_mcp}")
print(f"via @tool:       {via_tool}")
print(f"identical:       {via_mcp == via_tool}")

via MCP server:  ['2024-02-16', '2008-02-15']
via @tool:       ['2024-02-16', '2008-02-15']
identical:       True


**Step 1's output is the list above** - `["2024-02-16", "2008-02-15"]`, which is
what is wanted at this stage. Classification is separate.

## Step 2: classify by LLM reasoning

The design calls for a prompt that has an LLM *reason and classify*, so the
model produces the answer rather than calling a comparison tool. That invites a
confidently wrong answer, since a status label carries no evidence of how it was
reached.

Three guards:

| Guard | Effect |
|---|---|
| The model sees only the normalised dates, not the document | It cannot introduce a date that was never extracted |
| The prompt requires it to state its comparison | Faulty reasoning is visible rather than hidden behind a label |
| `classify_date` recomputes each answer | A wrong classification is detected with certainty |

The third is the only one that catches errors rather than reducing them. It is
detection, not prevention - prevention would mean not asking the model, which is
what is wanted.

**The reference date is fixed at 2024-01-01, not today.** The specification's own
example classifies 2024-02-16 as Upcoming, which is only true against that
reference.

In [6]:
normalised = normalized_with_context(found)
classified = classify(normalised, config=config)

print(json.dumps(to_output(classified), indent=2))

[
  {
    "original_text": "Distributed on Budget Day: 16 February 2024",
    "normalized_date": "2024-02-16",
    "status": "Upcoming",
    "reasoning": "2024-02-16 is later than 2024-01-01"
  },
  {
    "original_text": "Estate Duty does not apply to a person who dies after 15 February 2008",
    "normalized_date": "2008-02-15",
    "status": "Expired",
    "reasoning": "2008-02-15 is earlier than 2024-01-01"
  }
]


### The deterministic check

Diagnostic, not part of the answer. The tool computes what the status should be
and compares.

In [7]:
print(f"{'date':14s} {'model':10s} {'tool':10s} agrees  reasoning")
print("-" * 78)
for entry in check(classified):
    print(
        f"{entry['normalized_date']:14s} {entry['model_status']:10s} "
        f"{entry['tool_status']:10s} {str(entry['agrees']):6s}  {entry['reasoning'][:34]}"
    )

date           model      tool       agrees  reasoning
------------------------------------------------------------------------------
2024-02-16     Upcoming   Upcoming   True    2024-02-16 is later than 2024-01-0
2008-02-15     Expired    Expired    True    2008-02-15 is earlier than 2024-01


### Scored against known answers

In [8]:
print(score_dates(to_output(classified)).table())

field                        result detail
------------------------------------------------------------------------------
distribution                 Pass   2024-02-16 (Upcoming)
estate_duty                  Pass   2008-02-15 (Expired)

2/2 checks passed


## What "Ongoing" means

Expired and Upcoming are plain comparisons. Ongoing - "refers to a period that
is currently active" - cannot apply to a single date in the same way, so the
two cases need different rules.

**Single date:** Ongoing only when the date *is* the reference. A point has no
span to be inside.

**Period:** Ongoing when it has started and not yet finished, that is
`start <= reference <= end`. `classify_period` implements this.

**No period is extracted.** The specification asks for the document's dates,
and a span is not one of them, so `DocumentDates` captures the two single dates
only. The rule is still implemented and tested, because Ongoing is meaningless
without it - the cell below exercises it against FY2024, which the glossary
states as *"Financial Year 2024 is from 1 April 2024 to 31 March 2025"*, to
show the state changing.

In [9]:
from src.tools.date_tool import classify_period

# FY2024 against several reference dates, to show the state changing.
for reference in ("2024-01-01", "2024-04-01", "2024-09-30", "2025-03-31", "2025-04-01"):
    status = classify_period.invoke(
        {"start": "2024-04-01", "end": "2025-03-31", "reference": reference}
    )
    print(f"  reference {reference}: FY2024 is {status}")

  reference 2024-01-01: FY2024 is Upcoming
  reference 2024-04-01: FY2024 is Ongoing
  reference 2024-09-30: FY2024 is Ongoing
  reference 2025-03-31: FY2024 is Ongoing
  reference 2025-04-01: FY2024 is Expired


Against the required reference of 2024-01-01, FY2024 has not started, so a
span-aware classifier would call it Upcoming. It becomes Ongoing on 1 April 2024
and stays so for the whole year: a period is active for its full span, not only
at its start.

So Ongoing is implemented and verified, but it never appears in this pipeline's
output - no period is extracted, and even if FY2024 were, it would not be
Ongoing at the required reference date.

## Summary: the fields tested

| # | Field (the question) | Answer | Page | Status vs 2024-01-01 | Reasoning |
|---|---|---|---|---|---|
| 1 | Date the document was distributed | 2024-02-16 (from "16 February 2024") | 1 | Upcoming | 2024-02-16 is later than 2024-01-01 |
| 2 | Date named in the Estate Duty glossary entry | 2008-02-15 (from "15 February 2008") | 36 | Expired | 2008-02-15 is earlier than 2024-01-01 |

Both statuses verified by `classify_date` (see "Scored against known answers"
above, 2/2 checks passed).

Separately, `classify_period` (the period-aware classifier, not used by the
pipeline itself) was just exercised against FY2024 (1 April 2024 - 31 March
2025) at five reference dates, to demonstrate the Ongoing state.

Backed by 48 passing unit tests across `tests/test_date_reasoning.py`,
`tests/test_date_tool.py`, `tests/test_dates.py`, `tests/test_mcp_client.py`,
and `tests/test_mcp_server.py` - no network or model call required.

**Note:** the reasoning text above is simplified for readability; the
submitted `results/dates.json` records more verbose reasoning that compares
the month and day in the wrong direction, though the final status is still
correct.

## Conclusion

**Both dates extract, normalise and classify correctly,** and each agrees with
the deterministic check.

**On the two shapes:** a point date and a span are different things and need
different rules. `classify_date` compares a single date against the reference;
`classify_period` asks whether the reference falls inside a span. Only the
first is used by the pipeline - the second exists so that Ongoing is defined
rather than unreachable.

**On MCP:** the local server works and is what the pipeline uses. The `@tool`
decorator remains as an automatic fallback, and both return identical results
because they share one implementation - the server is a transport, not a second
copy of the logic.

**On the split:** the model finds dates in prose and classifies them; the tool
parses and checks. Each does the job it is suited to. A regex could replace the
finding step for this document, but would need extending for every new phrasing.

**On hallucination:** the check detects a wrong classification with certainty,
but detection is not prevention. It works here only because date comparison has
a deterministic answer to check against. A question like *"what are the key
revenue streams?"* has no such check - which is the harder problem Part 3 faces.

## Assumptions

- **An LLM finds the dates, rather than a regex.** Not strictly required - a
  pattern would return the same two dates today. Used because the specification
  says to extend the Part 1 solution, and because regexes do not scale: each new
  phrasing needs another pattern.
- **Every date is an explicit calendar date.** Open-ended expressions - "till
  present", "with immediate effect", "from 2008 onwards" - are out of scope.
  `normalize_date` returns None for anything it cannot parse into a concrete
  YYYY-MM-DD rather than inventing a boundary.
- **The reference date is fixed at 2024-01-01**, not today, so results stay
  stable over time.
- **Only the two cited dates are extracted.** The specification names them, and
  a span is not a date, so FY2024 is not a finding. `DocumentDates` captures
  distribution and estate duty only.
- **Ongoing is defined but never reached.** For a single date it would mean the
  date equals the reference, which neither does. The period rule lives in
  `classify_period` and is exercised above, so the state is implemented and
  tested rather than merely named.
- **Dates are read only from their cited pages** (1 and 36), bound by name in
  `config.yml` as in Part 1.
- **MCP runs locally over stdio.** Nothing is networked or deployed. Both `mcp`
  and `langchain-mcp-adapters` are MIT-licensed.
- **Temperature is 0**, so the same input yields the same classification.
- **Dates are written in full**, as the document's own style does ("16
  February 2024"). Further shapes are accepted:

  | Shape | Example | |
  |---|---|---|
  | Day month year | 16 February 2024 | The document's own form |
  | Month day, year | February 16, 2024 | Accepted |
  | ISO | 2024-02-16 | Accepted |
  | Slashed, day first | 16/02/2024 | Accepted, read day first - a US-style 02/16/2024 is misread rather than rejected |
  | Abbreviated month | 16 Feb 2024 | Accepted |
  | Ordinals, partial or non-English dates | 16th February 2024; "2024"; "16 Février" | Not parsed |